# Gain-prior + waveshaper on **LA2A** (SignalTrain) — coloration by composition

**Google Colab**: Runtime -> **GPU**. Open via *File -> Open notebook -> GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo for the
`08_la2a` + `06_output` modules and mounts Drive for the dataset. **Push local
changes before running.**

## What this is

A **retarget of `06_output/train_lstm_gain_prior_ws.ipynb`** (the Diff-SSL-G-Comp
waveshaper gain-prior run) to the **Teletronix LA-2A** (SignalTrain 1.1 dataset).
The model, training regime, and loss are the *same* `06_output` code, unchanged —
only the dataset, its split, and the conditioning width differ. Idea recap:

```
raw dry x ---------------------------------------------------┐
x*g  (amplitude-matched, g = 10^(gr/20)) --------------------┤
gr   (reduction-positive, ~[0,1]) ---------------------------┼- cat -> main LSTM(19->32)
tvcond cond_seq[16] (pool(|x|) (+) knobs[comp_limit,peak]) --┘          │
                                            ┌──────────────────┴─────────────────┐
                                   Δg = 12*tanh(lin_g(h))              c = lin_c(h)  (optional)
                                   s = x * 10^((gr + Δg)/20)                   │
                                   y = W(s) + c,   W(s) = s + r(s) - r(0)      │
```

Identity at init (zero-init heads + identity waveshaper), `W(0)=0` by
construction, `8,290` params (2 knobs instead of 4 -> 8,290 vs the diffssl run's
8,322; SOTA LSTM32TVC is ~8k). See `06_output/model_gainprior_ws.py`.

## Dataset — SignalTrain LA2A (`data/LA2A/all/`)

84 **long recordings** (4-20 min each, 44.1 kHz mono float), one per
`(Comp/Limit, Peak Reduction)` setting: `input_<id>_.wav` (dry) +
`target_<id>_LA2A_<Nc>__<cl>__<pr>.wav` (wet). **2 knobs** (`la2a_info.ini`):
Comp/Limit ∈ {0,1} switch, Peak Reduction ∈ {0,5,..,100}. 42 unique settings,
24.3 h total.

**GR is recomputed on-the-fly** from each `(dry, wet)` crop with the *exact*
export function — `src.dsp_torch.gain_reduction_db(dry, wet, 1024)` (causal
zero-left-padded 1024-RMS; the function `03_initial_GR_pred` used to write
`gr_curves/pair_*.pt`). A 1023-sample lookback fills the causal window from real
preceding samples, so each crop's GR is **bit-identical to slicing the full-file
curve** (verified: max |on-the-fly - .pt| ≈ 1.5e-5 dB, float32 vs float64). This
sidesteps the 18 GB `gr_curves/` tree entirely — only the 29 GB of WAVs are
cached.

## Split — temporal within each recording (`08_la2a/splits_la2a.py`)

Diff-SSL's song-level / `test_ground_truth` policy does **not** transfer: every
LA2A setting lives in only one (a few in two-three) recording(s), so holding out
whole recordings would delete a setting from training and break the knob
conditioning. Instead we split **temporally within each recording** — the
standard LA2A methodology (SignalTrain, Steinmetz TCN, Comparative-Study,
Optical-DRC all test on held-out *audio regions* at the same settings):

```
per recording:  [0, 0.8) -> train    [0.8, 0.9) -> val    [0.9, 1.0) -> test
```

- **Every setting is in all three splits** -> conditioning fully learnable; the
  test set probes generalisation to *unseen audio at known settings* (the
  question the LA-2A literature asks).
- **No content leakage** — boundaries are by time fraction, identical for every
  recording.
- Within each region, crops are taken at **evenly-spaced** offsets, capped at
  `crops_per_pair` (default 24/4/8 train/val/test) — the 24 h corpus is far too
  large to crop exhaustively; equal montage-spanning sampling per recording
  bounds the epoch while keeping settings balanced and material (music / speech /
  tones) represented. The manifest pins fracs + caps + inventory for exact reuse.

## Otherwise identical to the diffssl ws notebook (ablation contract)
tvcond knob conditioning, 3 s crops, `batch_size=16`, state reset per batch,
TBPTT sub-steps of 4410, AdamW + cosine, fixed 100-epoch budget, bf16 autocast,
and the 4c loss (`0.5*L1 + 0.5*MR-STFT_ext + 0.2*envelope-dB + 0.1*pre-emphasis`).
Reuses `system_gainprior.GainPriorSystem` and `model_gainprior_ws` **unchanged**.

In [1]:
# -- 0. Dependencies ---------------------------------------------------
# Uses nablafx (TVFiLMCond). Pin numpy first so lightning/nablafx installs
# can't downgrade Colab's numpy 2.x and break torch. Install lightning/nablafx
# --no-deps so they can't clobber Colab's CUDA torch. `rational` /
# `frechet_audio_distance` are nablafx import-chain deps we never use; stub
# both so `from nablafx...` doesn't drag in broken wheels.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 51.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 kB 24.2 MB/s eta 0:00:00
numpy 2.0.2, torch 2.11.0+cu128


In [2]:
# -- 1. Mount Drive (dataset) + clone repo from GitHub (code) ---------
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub -
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/LA2A"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module dirs: LA2A dataset/split (08_la2a) + the gain-prior model/system that
# it reuses unchanged (06_output). Both go on sys.path.
LA2A_DIR = os.path.join(REPO_ROOT, "08_la2a")
MODEL_DIR = os.path.join(REPO_ROOT, "06_output")
assert os.path.isfile(os.path.join(LA2A_DIR, "dataset_la2a.py")), (
    f"Clone failed or stale: {LA2A_DIR}. Did you push local changes?"
)
assert os.path.isfile(os.path.join(MODEL_DIR, "model_gainprior_ws.py")), (
    f"Missing 06_output model modules: {MODEL_DIR}"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "la2a_gain_prior_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "all")), (
    f"No all/ under {DATA_ROOT} - sync the SignalTrain LA2A WAVs to Drive first."
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep stale classes.
for _name in list(sys.modules):
    if _name in ("dataset_la2a", "splits_la2a", "model_tfilm",
                 "model_gainprior_ws", "system_gainprior", "amplitude_match"):
        del sys.modules[_name]

# repo root (for `src` + `nablafx`) + both module dirs
for p in (REPO_ROOT, os.path.join(REPO_ROOT, "nablafx"), MODEL_DIR, LA2A_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"LA2A_DIR   : {LA2A_DIR}")
print(f"MODEL_DIR  : {MODEL_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

Mounted at /content/drive
Cloning into '/content/Virtual-Analogue-Compressor-Modelling'...
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 200 (delta 19), reused 115 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (200/200), 95.66 MiB | 15.96 MiB/s, done.
Resolving deltas: 100% (19/19), done.
REPO_ROOT  : /content/Virtual-Analogue-Compressor-Modelling
LA2A_DIR   : /content/Virtual-Analogue-Compressor-Modelling/08_la2a
MODEL_DIR  : /content/Virtual-Analogue-Compressor-Modelling/06_output
DATA_ROOT  : /content/drive/Othercomputers/MacBook Air/data/LA2A
OUTPUT_DIR : /content/drive/Othercomputers/MacBook Air/data/la2a_gain_prior_runs


In [3]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# Only the input/target WAVs are cached (~29 GB float32). The 18 GB gr_curves/
# tree is deliberately NOT needed: dataset_la2a recomputes GR on-the-fly per
# crop, bit-identical to the exported .pt (see its docstring). One-time copy per
# session; subsequent reads are fast local SSD.

import shutil
from dataset_la2a import discover_la2a_pairs

LOCAL_DATA_ROOT = "/content/LA2A"
pairs = discover_la2a_pairs(DATA_ROOT)
print(f"Caching {len(pairs)} recordings (input+target WAVs) -> {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

for i, p in enumerate(pairs, 1):
    for key in ("dry", "wet"):
        _mirror(p[key], Path(LOCAL_DATA_ROOT) / Path(p[key]).relative_to(DATA_ROOT))
    if i % 10 == 0 or i == len(pairs):
        print(f"  cached {i}/{len(pairs)}")

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

Caching 84 recordings (input+target WAVs) -> /content/LA2A
  cached 10/84
  cached 20/84
  cached 30/84
  cached 40/84
  cached 50/84
  cached 60/84
  cached 70/84
  cached 80/84
  cached 84/84
Using local cache: /content/LA2A


In [4]:
# -- 3. Imports & hyper-parameters (waveshaper gain-prior + 4c loss) --

import importlib
import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset_la2a as _dataset_la2a
importlib.reload(_dataset_la2a)
from dataset_la2a import (
    BATCH_SIZE, RMS_WINDOW, SAMPLE_LENGTH, SAMPLE_RATE,
    La2aCropDataModule, discover_la2a_pairs,
)

import splits_la2a as _splits_la2a
importlib.reload(_splits_la2a)
from splits_la2a import (
    LA2A_PARAM_ORDER, LA2A_PARAM_RANGES, build_la2a_split_manifest,
)

import model_tfilm as _model_tfilm          # model_gainprior_ws imports from it
importlib.reload(_model_tfilm)
import model_gainprior_ws as _model_gainprior_ws
importlib.reload(_model_gainprior_ws)
from model_gainprior_ws import GainPriorWSDiffSSLLSTM

import system_gainprior as _system_gainprior   # reused UNCHANGED from the diffssl nb
importlib.reload(_system_gainprior)
from system_gainprior import GainPriorSystem

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split: temporal within each recording (every setting in every split) --
SPLIT_SEED  = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.8, 0.1, 0.1
# evenly-spaced crops per recording per split; caps the 24 h corpus to a
# trainable epoch. Raise for more data / more epochs of compute.
CROPS_PER_PAIR = {"train": 24, "val": 4, "test": 8}

# -- training (same diffssl TBPTT regime as the ws gain-prior nb) --
LR               = 1e-3
MAX_EPOCHS       = 100      # fixed budget == cosine T_max
STEP_NUM_SAMPLES = 4410     # diffssl TBPTT sub-step (0.1 s)
SCHEDULER        = "cosine"
ETA_MIN          = 1e-6
USE_AMP          = True
CHECK_VAL_EVERY_N_EPOCH = 1

# -- model core (identical to the diffssl ws run, except the 2 LA2A knobs) --
HIDDEN_SIZE     = 32
NUM_LAYERS      = 1
NUM_CONTROLS    = 2      # LA2A: [comp_limit, peak_reduction]
TVCOND_DIM      = 16
COND_BLOCK_SIZE = 128
COND_NUM_LAYERS = 1
DELTA_MAX_DB    = 12.0   # bound on the learned gain correction (dB)
USE_COLOR       = True   # keep the additive head; the WS/color split in cell 9
                         # shows which mechanism carries the residual

# -- waveshaper (the coloration mechanism) --
WS_HIDDEN = 8            # transfer-curve MLP width (1->8->8->1)
WS_FILM   = False        # True: FiLM the curve from the LSTM state (zero-init)

# -- loss (4c, unchanged). ENV_WEIGHT=0, PE_WEIGHT=0, MRSTFT_VARIANT="sota" == exact 02b --
TD_WEIGHT      = 0.5      # L1
FD_WEIGHT      = 0.5      # MR-STFT
ENV_WEIGHT     = 0.2      # RMS-envelope L1 in dB (differentiable GR MAE, window 1024)
PE_WEIGHT      = 0.1      # pre-emphasis L1 (transients)
MRSTFT_VARIANT = "extended"   # "extended" (adds 4096-FFT + lin-mag) | "sota"

RUN_TAG    = "la2a_lstm32_gain_prior_ws"
RESUME_RUN = None

NVIDIA L4


In [5]:
# -- 4. Preview split - temporal within each recording ----------------
# Every (comp_limit, peak_reduction) setting appears in train/val/test on
# DISJOINT temporal regions of its recording: conditioning is fully learnable
# and the test set is unseen audio at known settings (the LA2A convention).

pairs = discover_la2a_pairs(DATA_ROOT)
preview = build_la2a_split_manifest(
    pairs, seed=SPLIT_SEED, sample_length=SAMPLE_LENGTH,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
    crops_per_pair=CROPS_PER_PAIR,
)

print(f"Recordings : {len(preview.pairs)}")
print(f"Settings   : {len(preview.settings)} unique [comp_limit, peak_reduction]")
print(f"  comp/limit values : {sorted({s[0] for s in preview.settings})}")
print(f"  peak_reduction    : {sorted({s[1] for s in preview.settings})}")
print(f"Regions    : train[0,{TRAIN_FRAC}) val[{TRAIN_FRAC},{round(TRAIN_FRAC+VAL_FRAC,3)}) "
      f"test[{round(TRAIN_FRAC+VAL_FRAC,3)},1) of every recording")
print(f"Crop totals: {preview.crop_counts}  (<= {CROPS_PER_PAIR} per recording)")
mins = {k: v * SAMPLE_LENGTH / SAMPLE_RATE / 60 for k, v in preview.crop_counts.items()}
print("Audio (min): " + "  ".join(f"{k}={mins[k]:.1f}" for k in ("train", "val", "test")))

assert all(preview.crop_counts[k] > 0 for k in ("train", "val", "test")), "empty split!"
print("\nExample recordings (id, knobs, duration):")
for p in preview.pairs[:3] + preview.pairs[-2:]:
    print(f"  id {p['id']:>3}  cl={p['comp_limit']} pr={p['peak_reduction']:>3}  "
          f"{p['frames'] / SAMPLE_RATE:.0f}s")

Recordings : 84
Settings   : 42 unique [comp_limit, peak_reduction]
  comp/limit values : [0, 1]
  peak_reduction    : [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
Regions    : train[0,0.8) val[0.8,0.9) test[0.9,1) of every recording
Crop totals: {'train': 2016, 'val': 336, 'test': 672}  (<= {'train': 24, 'val': 4, 'test': 8} per recording)
Audio (min): train=100.8  val=16.8  test=33.6

Example recordings (id, knobs, duration):
  id 138  cl=0 pr=  0  1200s
  id 139  cl=0 pr=  5  1200s
  id 140  cl=0 pr= 10  1200s
  id 262  cl=1 pr= 95  900s
  id 263  cl=1 pr=100  900s


In [6]:
# -- 5. Model size ----------------------------------------------------

model = GainPriorWSDiffSSLLSTM(
    num_controls=NUM_CONTROLS, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    tvcond_dim=TVCOND_DIM, cond_block_size=COND_BLOCK_SIZE, cond_num_layers=COND_NUM_LAYERS,
    delta_max_db=DELTA_MAX_DB, use_color=USE_COLOR,
    ws_hidden=WS_HIDDEN, ws_film=WS_FILM,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"GainPriorWSDiffSSLLSTM: {n_params:,} params  "
      f"(hidden={HIDDEN_SIZE}, tvcond_dim={TVCOND_DIM}, controls={NUM_CONTROLS}, "
      f"delta_max={DELTA_MAX_DB} dB, color={USE_COLOR}, "
      f"ws_hidden={WS_HIDDEN}, ws_film={WS_FILM})")
for name, mod in model.named_children():
    print(f"  {name:10s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\nDiffssl ws run was 8,322 (4 knobs); LA2A has 2 knobs -> {n_params:,}. "
      f"SOTA LSTM32TVC is 8k - still parameter-matched.")
print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | {SAMPLE_RATE} Hz | "
      f"TBPTT step {STEP_NUM_SAMPLES} | tvcond block {COND_BLOCK_SIZE}")

GainPriorWSDiffSSLLSTM: 8,290 params  (hidden=32, tvcond_dim=16, controls=2, delta_max=12.0 dB, color=True, ws_hidden=8, ws_film=False)
  cond_nn    1,344
  lstm       6,784
  lin_gain   33
  lin_color  33
  waveshaper 96

Diffssl ws run was 8,322 (4 knobs); LA2A has 2 knobs -> 8,290. SOTA LSTM32TVC is 8k - still parameter-matched.
Crop 132300 (3.00s) | 44100 Hz | TBPTT step 4410 | tvcond block 128


In [7]:
# -- 6. DataModule + zero-init sanity check ---------------------------
# Zero-init linear heads + identity-init waveshaper make the untrained model
# IDENTICAL to the amplitude-matched baseline (dry x 10^(gr/20)). Verify on a
# real LA2A batch before training: the run must START from that baseline.

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"la2a_gain_prior_ws_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = La2aCropDataModule(
    data_root=DATA_ROOT, sample_length=SAMPLE_LENGTH, sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE, split_seed=SPLIT_SEED,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
    crops_per_pair=CROPS_PER_PAIR, rms_window=RMS_WINDOW,
    split_manifest_path=split_path, num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")

# -- zero-init sanity: untrained model == amplitude match --------------
from amplitude_match import amplitude_match

if not RESUME_RUN:
    _dry, _gr, _wet, _p = next(iter(dm.val_dataloader()))
    with torch.no_grad():
        model.reset_states()
        _y0 = model(_dry, _gr, _p)
    _diff = float((_y0 - amplitude_match(_dry, _gr)).abs().max())
    _l1 = float(torch.nn.functional.l1_loss(_y0, _wet))
    print(f"zero-init |model - amplitude_match| max = {_diff:.2e}")
    assert _diff < 1e-5, "gain-prior heads / waveshaper are not identity-initialised!"
    print(f"untrained (== amp-match) crop L1 vs wet: {_l1:.6f}  <- training starts here")
    model.reset_states()
    del _dry, _gr, _wet, _p, _y0

DataLoader num_workers: 8
NEW run: la2a_gain_prior_ws_20260705_113527_la2a_lstm32_gain_prior_ws
Recordings     : 84
Settings       : 42 unique (comp_limit, peak_reduction)
Split fractions: train=0.8 val=0.1 test=0.1
Crops per rec  : {'train': 24, 'val': 4, 'test': 8}
Crop totals    : {'train': 2016, 'val': 336, 'test': 672}
La2aCropDataset[train]: 2016 crops from 84 recordings  [<= 24/rec, sample_length=132300, 100.8 min audio]
La2aCropDataset[val]: 336 crops from 84 recordings  [<= 4/rec, sample_length=132300, 16.8 min audio]
La2aCropDataset[test]: 672 crops from 84 recordings  [<= 8/rec, sample_length=132300, 33.6 min audio]
Train/val/test crops: 2016 / 336 / 672
Batches/epoch (train): 126  (batch_size=16)
zero-init |model - amplitude_match| max = 0.00e+00
untrained (== amp-match) crop L1 vs wet: 0.064955  <- training starts here


In [8]:
# -- 7. Train ---------------------------------------------------------

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "gain_prior_ws: y = W(x * 10^((gr + delta_g)/20)) + color, W identity-init",
        "model_type": "GainPriorWSDiffSSLLSTM",
        "model_ref": "diffssl ws gain-prior (06_output/train_lstm_gain_prior_ws) retargeted to LA2A",
        "dataset": "SignalTrain-LA2A",
        "setting": "multi (all 42 comp_limit x peak_reduction settings; tvcond on 2 knobs)",
        "conditioning": "knobs via tvcond (TVFiLMCond); GR as multiplicative prior input",
        "gr_source": "on-the-fly gain_reduction_db(dry, wet, 1024) == exported gr_curves/*.pt",
        "sample_rate": SAMPLE_RATE, "sample_length": SAMPLE_LENGTH, "batch_size": BATCH_SIZE,
        "rms_window": RMS_WINDOW, "step_num_samples": STEP_NUM_SAMPLES,
        "param_order": LA2A_PARAM_ORDER, "param_ranges": LA2A_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "split_policy": "temporal_within_recording (every setting in train/val/test; test = unseen audio regions)",
        "split_fracs": {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": TEST_FRAC},
        "crops_per_pair": CROPS_PER_PAIR,
        "num_settings": len(dm.split.settings), "num_recordings": len(dm.split.pairs),
        "crop_counts": dm.split.crop_counts,
        "model": {"hidden_size": HIDDEN_SIZE, "num_layers": NUM_LAYERS,
                   "num_controls": NUM_CONTROLS, "tvcond_dim": TVCOND_DIM,
                   "cond_block_size": COND_BLOCK_SIZE, "cond_num_layers": COND_NUM_LAYERS,
                   "delta_max_db": DELTA_MAX_DB, "use_color": USE_COLOR,
                   "ws_hidden": WS_HIDDEN, "ws_film": WS_FILM,
                   "num_params": n_params},
        "loss": {"td_weight": TD_WEIGHT, "fd_weight": FD_WEIGHT,
                 "env_weight": ENV_WEIGHT, "pe_weight": PE_WEIGHT,
                 "mrstft_variant": MRSTFT_VARIANT,
                 "kind": "td*L1 + fd*MRSTFT + env*envdB_L1 + pe*preemph_L1"},
        "metrics": ["esr", "rmse", "mae", "mse"],
        "optimizer": f"adamw + {SCHEDULER}",
        "scheduler": SCHEDULER, "eta_min": ETA_MIN, "use_amp": USE_AMP,
        "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
        "training": "diffssl_crop_batches + tbptt_substeps (reset each batch)",
        "lr": LR, "max_epochs": MAX_EPOCHS,
    }, f, indent=2)

system = GainPriorSystem(
    model=model, lr=LR, step_num_samples=STEP_NUM_SAMPLES,
    td_weight=TD_WEIGHT, fd_weight=FD_WEIGHT,
    env_weight=ENV_WEIGHT, pe_weight=PE_WEIGHT, mrstft_variant=MRSTFT_VARIANT,
    scheduler=SCHEDULER, max_epochs=MAX_EPOCHS, eta_min=ETA_MIN, use_amp=USE_AMP,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers, log_every_n_steps=10,
    check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model  │ GainPriorWSDiffSSLLSTM  │  8.3 K │ train │     0 │
│ 1 │ l1     │ L1Loss                  │      0 │ train │     0 │
│ 2 │ mrstft │ MultiResolutionSTFTLoss │      0 │ train │     0 │
└───┴────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 8.3 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.3 K                                                                                                
Total estimated model params size (MB): 0.033                                                                      
Modules in train mode: 38                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

: 

In [ ]:
# -- 8. Test (held-out temporal regions - unseen audio at known settings) --

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, datamodule=dm, ckpt_path=best_ckpt)

In [ ]:
# -- 9. Plot: prediction vs target + head shares ----------------------
# For each example: (top) waveform overlay, (bottom) GR input vs corrected
# gain gr+delta. Titles split the residual into the waveshaper share
# (mean|W(s)-s|, phase-locked harmonics) and the additive share (mean|color|,
# whatever cannot be a function of the instantaneous gained sample).

import matplotlib.pyplot as plt
import numpy as np
from system_gainprior import esr_metric

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_batches = list(dm.val_dataloader())
dry, gr, wet, params = val_batches[len(val_batches) // 2]

with torch.no_grad():
    system.model.reset_states()
    pred, delta_db, color, ws_res = system.model(
        dry.cuda(), gr.cuda(), params.cuda(), return_parts_full=True)
    pred, delta_db = pred.cpu(), delta_db.cpu()
    color, ws_res = color.cpu(), ws_res.cpu()

dry_np, wet_np, pred_np = dry.numpy(), wet.numpy(), pred.numpy()
gr_np, delta_np = gr.numpy(), delta_db.numpy()
color_np, ws_np = color.numpy(), ws_res.numpy()
n_plots = min(3, dry_np.shape[0])
fig, axes = plt.subplots(2 * n_plots, 1, figsize=(14, 4.6 * n_plots), squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for r in range(n_plots):
    ax = axes[2 * r, 0]
    ax.plot(t, dry_np[r, 0], label="Dry", alpha=0.35, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    mae_r = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    ax.set_title(f"crop {r} - MAE {mae_r:.4f} | ESR {float(esr_metric(tv, pv)):.4f} | "
                 f"mean|dg| {np.abs(delta_np[r]).mean():.3f} dB | "
                 f"mean|ws| {np.abs(ws_np[r]).mean():.5f} | "
                 f"mean|color| {np.abs(color_np[r]).mean():.5f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)

    ax = axes[2 * r + 1, 0]
    ax.plot(t, gr_np[r, 0], label="GR input (dB)", lw=0.7, color="#1f77b4")
    ax.plot(t, gr_np[r, 0] + delta_np[r, 0], label="GR + learned delta", lw=0.7,
            color="#d62728", alpha=0.8)
    ax.set_ylabel("gain (dB)"); ax.legend(loc="lower right", fontsize=8)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"LA2A waveshaper gain-prior LSTM - best val loss {callbacks[0].best_model_score:.6f}", y=1.002)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()

In [ ]:
# -- 10. Learned transfer curve ---------------------------------------
# The waveshaper is where harmonics come from (composition): plot W(s)
# against identity plus the deviation. Should bow away from identity at high
# |s| (saturation); asymmetry = even harmonics. With WS_FILM=True this is the
# UNMODULATED static curve (h=None path).

s_sweep = torch.linspace(-1.0, 1.0, 1001).view(1, 1, -1).cuda()
with torch.no_grad():
    w_sweep = system.model.waveshaper(s_sweep).cpu().flatten().numpy()
s_np = s_sweep.cpu().flatten().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(s_np, s_np, "--", color="gray", lw=0.8, label="identity")
ax1.plot(s_np, w_sweep, color="#d62728", lw=1.2, label="W(s)")
ax1.set_xlabel("in"); ax1.set_ylabel("out"); ax1.set_title("Learned transfer curve")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax2.plot(s_np, w_sweep - s_np, color="#d62728", lw=1.2)
ax2.set_xlabel("in"); ax2.set_ylabel("W(s) - s")
ax2.set_title(f"Deviation from identity (max {np.abs(w_sweep - s_np).max():.4f})")
ax2.grid(alpha=0.3)
fig.tight_layout()
curve_path = os.path.join(RUN_DIR, "eval_waveshaper_curve.png")
fig.savefig(curve_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {curve_path}")
plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb\"